# Weekly Report Generator

Runs every Monday. Analyses the previous Mon–Sun and writes a single `.md` file to `analysis/`.

**Run All** — no manual config needed.

In [0]:
# ── Setup ─────────────────────────────────────────────────────────

import sys
sys.path.append('/Workspace/Users/davidgao734@gmail.com/boba-cafe/POS')
sys.path.append('/Workspace/Users/davidgao734@gmail.com/boba-cafe/weekly-analysis')

from datetime import datetime, timedelta
from pipeline.transforms import _get_spark

spark = _get_spark()

# Always analyse previous Mon–Sun
today      = datetime.now()
this_mon   = today - timedelta(days=today.weekday())
week_end   = (this_mon - timedelta(days=1)).strftime("%Y-%m-%d")   # last Sunday
week_start = (this_mon - timedelta(days=7)).strftime("%Y-%m-%d")  # last Monday
prior_end  = (this_mon - timedelta(days=8)).strftime("%Y-%m-%d")  # Sunday before that
prior_start= (this_mon - timedelta(days=14)).strftime("%Y-%m-%d") # Monday before that

print(f"Current week : {week_start} → {week_end}")
print(f"Prior week   : {prior_start} → {prior_end}")

In [0]:
# ── Load Data ─────────────────────────────────────────────────────

from config import TRANSACTIONS_TABLE, DAILY_SALES_TABLE, HIERARCHY_CSV, ANALYSIS_DIR
from config import LOW_SALES_PCT, LOW_CASH_DROP_PCT, SALES_GAP_MINUTES, TAPIOCA_GAP_MINUTES, TAPIOCA_KEYWORD, MIN_TRADING_REVENUE
from modules.loader import load_transactions, load_daily_sales, load_product_hierarchy

cur_txn   = load_transactions(spark, TRANSACTIONS_TABLE, week_start, week_end)
pri_txn   = load_transactions(spark, TRANSACTIONS_TABLE, prior_start, prior_end)
cur_sales = load_daily_sales(spark, DAILY_SALES_TABLE, week_start, week_end)
pri_sales = load_daily_sales(spark, DAILY_SALES_TABLE, prior_start, prior_end)
hierarchy = load_product_hierarchy(HIERARCHY_CSV)

ANOMALY_CFG = {
    "LOW_SALES_PCT":       LOW_SALES_PCT,
    "LOW_CASH_DROP_PCT":   LOW_CASH_DROP_PCT,
    "SALES_GAP_MINUTES":   SALES_GAP_MINUTES,
    "TAPIOCA_GAP_MINUTES": TAPIOCA_GAP_MINUTES,
    "TAPIOCA_KEYWORD":     TAPIOCA_KEYWORD,
    "MIN_TRADING_REVENUE": MIN_TRADING_REVENUE,
}

print(f"Transactions  this week  : {len(cur_txn):,} rows")
print(f"Transactions  prior week : {len(pri_txn):,} rows")
print(f"Daily sales   this week  : {len(cur_sales):,} rows")
print(f"Product hierarchy loaded : {'yes' if not hierarchy.empty else 'no — using raw product names'}")

In [0]:
# ── Build Report ──────────────────────────────────────────────────

from modules import sales, basket, product_health, anomaly

header = f"""# Weekly Report: {week_start} to {week_end}

_Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}_

---
"""

sections = [
    header,
    sales.build(cur_sales, pri_sales),
    "\n---\n",
    basket.build(cur_txn, pri_txn),
    "\n---\n",
    product_health.build(cur_txn, pri_txn, hierarchy),
    "\n---\n",
    anomaly.build(cur_txn, spark, TRANSACTIONS_TABLE, week_start, week_end, ANOMALY_CFG),
]

report_md = "\n".join(sections)
print("Report built successfully")

In [0]:
# ── Save to analysis/ folder ──────────────────────────────────────

import os

os.makedirs(ANALYSIS_DIR, exist_ok=True)
filename  = f"{week_start}_weekly_report.md"
out_path  = os.path.join(ANALYSIS_DIR, filename)

with open(out_path, "w", encoding="utf-8") as f:
    f.write(report_md)

print(f"Saved → {out_path}")
print()
# Preview first 50 lines
for line in report_md.splitlines()[:50]:
    print(line)